# Historical Pattern Analysis (10-15 Years)

Analyze historical patterns, volume spikes, and news correlation for NSE stocks

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

repo_root = Path('..').resolve()
sys.path.append(str(repo_root / 'services' / 'api'))

from app.live_data import LiveDataService, PatternRecognition
from app.ml_predictor import StockPredictor

In [ ]:
# Initialize services
live_service = LiveDataService()
pattern_service = PatternRecognition()
predictor = StockPredictor()

# Fetch 15 years of data
symbol = 'RELIANCE'
df = live_service.get_historical_data(symbol, '15y')
print(f"Data points: {len(df)}")
df.tail()

In [ ]:
# Detect all patterns in historical data
patterns = pattern_service.detect_patterns(df)
print(f"Total patterns detected: {len(patterns)}")

# Pattern distribution
pattern_counts = {}
for p in patterns:
    pattern_counts[p.pattern] = pattern_counts.get(p.pattern, 0) + 1

print("\nPattern Distribution:")
for pattern, count in sorted(pattern_counts.items(), key=lambda x: x[1], reverse=True):
    print(f"{pattern}: {count}")

In [ ]:
# Volume analysis - detect news correlation
df['volume_spike'] = df['Volume'] > df['Volume'].rolling(20).mean() * 2
df['price_change'] = df['Close'].pct_change()
df['big_move'] = abs(df['price_change']) > 0.03  # 3% move

# News proxy: volume spike + big price move
df['news_event'] = df['volume_spike'] & df['big_move']

news_events = df[df['news_event']]
print(f"\nPotential news events detected: {len(news_events)}")
print(f"Average price change on news days: {news_events['price_change'].mean()*100:.2f}%")
print(f"Average volume spike: {(news_events['Volume'] / df['Volume'].mean()).mean():.2f}x")

In [ ]:
# Train ML model
training_result = predictor.train_model(df)
print("\nModel Training Results:")
print(f"Train Accuracy: {training_result['train_accuracy']*100:.2f}%")
print(f"Test Accuracy: {training_result['test_accuracy']*100:.2f}%")

In [ ]:
# Make prediction for next day
prediction = predictor.predict(df)
print("\nNext Day Prediction:")
print(f"Direction: {prediction['direction'].upper()}")
print(f"Confidence: {prediction['confidence']*100:.1f}%")
print(f"Probability Up: {prediction['probability_up']*100:.1f}%")
print(f"Probability Down: {prediction['probability_down']*100:.1f}%")

In [ ]:
# Visualize patterns over time
plt.figure(figsize=(15, 8))

plt.subplot(3, 1, 1)
plt.plot(df.index, df['Close'], label='Close Price', color='blue')
plt.title(f'{symbol} - 15 Year Price History')
plt.ylabel('Price')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(3, 1, 2)
plt.bar(df.index, df['Volume'], label='Volume', color='gray', alpha=0.5)
plt.bar(news_events.index, news_events['Volume'], label='News Events', color='red', alpha=0.7)
plt.ylabel('Volume')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(3, 1, 3)
plt.plot(df.index, df['price_change']*100, label='Daily Return %', color='green', alpha=0.6)
plt.axhline(y=0, color='black', linestyle='--', alpha=0.3)
plt.ylabel('Return %')
plt.xlabel('Date')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Pattern performance analysis
print("\nPattern Performance Analysis:")
print("Analyzing how patterns performed historically...\n")

# Analyze hammer pattern success rate
features = predictor.prepare_features(df)
hammer_days = df[features['hammer'] == 1]
if len(hammer_days) > 0:
    hammer_success = (hammer_days['Close'].shift(-1) > hammer_days['Close']).mean()
    print(f"Hammer Pattern Success Rate: {hammer_success*100:.1f}%")
    print(f"Hammer occurrences: {len(hammer_days)}")

# Analyze doji pattern
doji_days = df[features['doji'] == 1]
if len(doji_days) > 0:
    doji_up = (doji_days['Close'].shift(-1) > doji_days['Close']).mean()
    print(f"\nDoji Pattern - Next Day Up: {doji_up*100:.1f}%")
    print(f"Doji occurrences: {len(doji_days)}")